# Hy3 API - 工具调用示例

本示例演示如何使用 Hy3 API 进行工具调用，包括单次工具调用和多轮工具循环。

In [ ]:
from openai import OpenAI
import os
import json
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(
    api_key=os.getenv("API_KEY", "EMPTY"),
    base_url=os.getenv("BASE_URL", "http://127.0.0.1:8000/v1")
)


## 工具定义

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的实时天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "执行数学计算",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "数学表达式"},
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "获取指定股票的当前价格",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {"type": "string", "description": "股票代码"},
                },
                "required": ["symbol"],
            },
        },
    },
]

def mock_tool_call(tool_name, arguments):
    if tool_name == "get_weather":
        city = arguments.get("city", "")
        return f'{{"city": "{city}", "temperature": 25, "condition": "晴", "humidity": 60}}'
    elif tool_name == "calculate":
        expr = arguments.get("expression", "")
        try:
            result = eval(expr)
            return f'{{"expression": "{expr}", "result": {result}}}'
        except:
            return f'{{"expression": "{expr}", "result": "计算错误"}}'
    elif tool_name == "get_stock_price":
        symbol = arguments.get("symbol", "")
        return f'{{"symbol": "{symbol}", "price": 150.50, "change": "+2.3%"}}'
    return f'{{"error": "unknown tool: {tool_name}"}}'

## 单次工具调用

In [ ]:
response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": "北京今天天气怎么样？"}],
    tools=tools,
)

print(f"响应 ID: {response.id}")
print(f"finish_reason: {response.choices[0].finish_reason}")

message = response.choices[0].message
if message.tool_calls:
    for tc in message.tool_calls:
        print(f"\n工具调用:")
        print(f"  name: {tc.function.name}")
        print(f"  arguments: {tc.function.arguments}")
        
        args = json.loads(tc.function.arguments)
        result = mock_tool_call(tc.function.name, args)
        print(f"  工具执行结果: {result}")

## 多轮工具循环

In [ ]:
messages = [
    {"role": "user", "content": "帮我计算一下：100乘以20%，然后用这个结果查询苹果公司的股票价格"},
]

max_rounds = 5
round_count = 0

print(f"初始问题: {messages[0]['content']}")

while round_count < max_rounds:
    def get_role(msg):
        if isinstance(msg, dict):
            return msg.get("role")
        return msg.role
    round_count += 1
    print(f"\n--- 第 {round_count} 轮 ---")

    response = client.chat.completions.create(
        model="hy3",
        messages=messages,
        tools=tools,
    )

    choice = response.choices[0]
    message = choice.message

    if choice.finish_reason == "tool_calls" and message.tool_calls:
        for tc in message.tool_calls:
            print(f"  工具调用: {tc.function.name}({tc.function.arguments})")

            args = json.loads(tc.function.arguments)
            tool_result = mock_tool_call(tc.function.name, args)
            print(f"  工具返回: {tool_result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result,
            })
    else:
        print(f"  最终回复: {message.content}")
        break

print(f"\n总轮数: {round_count}")
tool_count = len([m for m in messages if get_role(m) == "tool"])
print(f"  工具调用次数: {tool_count}")

## 关键要点

1. **工具定义**：必须包含 `type`、`function.name`、`function.description` 和 `function.parameters`
2. **finish_reason**：工具调用时为 `"tool_calls"`，最终回复时为 `"stop"`
3. **工具结果**：必须以 `role: "tool"` 的形式加入对话历史
4. **tool_call_id**：必须与工具调用时的 `id` 对应
5. **安全性**：执行工具前应验证参数，避免注入攻击